#  Análise de Desempenho — Itens de Venda (Caixa)

**Squad 3 — Batch Lojas Físicas | Luiz Henrique Portácio**

Este notebook apresenta os indicadores de desempenho dos itens
vendidos nas lojas físicas para consumo pelos gestores da operação.

**Estrutura:**
1. Análise exploratória (shape, schema, nulos).
2. **Qualidade de Dados** — integridade por regra técnica.
3. **Indicadores de Negócio** — KPIs 6-10 com gráficos interativos
   (Plotly) e estáticos (Matplotlib).

> **Nota:** Dashboard oficial via **Looker**, conectado a
> `squad3.gold_physical_itens_venda_caixa`.

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

In [0]:
from pyspark.sql.functions import (
    col, sum as spark_sum, count, when,
    round as spark_round, avg as spark_avg,
    countDistinct
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import plotly.graph_objects as go

# ── Paleta de cores (mesmo padrão do notebook de lojas) ──────────────────────
COR_POSITIVO = "#1B5E20"
COR_NEGATIVO = "#B71C1C"
COR_ALERTA   = "#E65100"
COR_AZUL     = "#0D47A1"
COR_CINZA    = "#424242"

PALETA_DISCRETA = [
    "#1565C0", "#2E7D32", "#6A1B9A", "#E65100",
    "#00695C", "#AD1457", "#4527A0", "#37474F",
]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#FAFAFA",
    "axes.edgecolor":   "#BDBDBD",
    "axes.labelcolor":  "#212121",
    "text.color":       "#212121",
    "xtick.color":      "#424242",
    "ytick.color":      "#424242",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "font.size":        10,
})

LAYOUT_BASE = dict(
    plot_bgcolor="white",
    paper_bgcolor="white",
    hoverlabel=dict(bgcolor="#212121", font_color="white", font_size=12),
)

def formatar_reais(valor):
    if valor is None or valor != valor:
        return "—"
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def card_minimalista(fig, n_total, idx, pct, valor_txt,
                     status_txt, cor, titulo, detalhe,
                     fontsize_valor=36):
    """Desenha um score card minimalista (linha no topo, fundo branco)."""
    ax = fig.add_subplot(1, n_total, idx)
    ax.set_facecolor("white")
    ax.axis("off")
    # Linha colorida no topo
    ax.plot([0.08, 0.92], [0.97, 0.97], color=cor,
            linewidth=5, solid_capstyle="butt",
            transform=ax.transAxes, clip_on=False)
    # Valor principal
    ax.text(0.5, 0.68, valor_txt, ha="center", va="center",
            fontsize=fontsize_valor, fontweight="bold", color=cor,
            transform=ax.transAxes)
    # Status
    ax.text(0.5, 0.50, status_txt, ha="center", va="center",
            fontsize=9, fontweight="bold", color=cor,
            transform=ax.transAxes)
    # Separador
    ax.plot([0.10, 0.90], [0.42, 0.42], color="#E0E0E0",
            linewidth=0.8, transform=ax.transAxes, clip_on=False)
    # Título
    ax.text(0.5, 0.30, titulo, ha="center", va="center",
            fontsize=9, color="#424242", fontweight="bold",
            transform=ax.transAxes)
    # Detalhe
    ax.text(0.5, 0.12, detalhe, ha="center", va="center",
            fontsize=8, color="#757575",
            transform=ax.transAxes)
    # Borda sutil
    for side in ["top", "bottom", "left", "right"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#E0E0E0")
        ax.spines[side].set_linewidth(0.8)
    return ax

---
##  Sumário Executivo

In [0]:
df_gold = read_delta(GOLD_ITENS_VENDA_CAIXA_PATH, adls_options)

anos         = sorted([r.ano for r in df_gold.select("ano").distinct().collect()])
ano_ref      = max(anos) if anos else None
ano_ini      = min(anos) if anos else None
periodo_hist = f"{ano_ini}–{ano_ref}" if (ano_ini and ano_ref and ano_ini != ano_ref) else str(ano_ref)
df_ref       = df_gold.filter(col("ano") == ano_ref)

total_itens      = df_ref.count()
total_transacoes = df_ref.agg(spark_sum("qtd_transacoes")).collect()[0][0] or 0
receita_total    = df_ref.agg(spark_sum("receita_produto_loja_mes")).collect()[0][0] or 0
ticket_medio     = df_ref.filter(col("media_itens_transacao").isNotNull()).agg(
    spark_avg("media_itens_transacao")).collect()[0][0] or 0
pct_feriado      = 0
feriado_row      = df_ref.filter(col("venda_em_feriado") == True).agg(
    spark_sum("receita_produto_loja_mes")).collect()[0][0]
if feriado_row and receita_total:
    pct_feriado = round(feriado_row / receita_total * 100, 1)

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║     SUMÁRIO EXECUTIVO — ITENS DE VENDA                           ║
║     Período histórico : {periodo_hist:<41}║
║     Ano de referência : {str(ano_ref):<41}║
╠══════════════════════════════════════════════════════════════════╣
║  📦  Total de itens vendidos  :  {total_itens:>15,}               ║
║  🛒  Total de transações      :  {total_transacoes:>15,}               ║
║  💰  Receita total ({ano_ref})  :  {formatar_reais(receita_total):<30}║
║  📋  Média de itens/transação :  {ticket_medio:>14.1f}                ║
║  🗓️  Receita em feriados       :  {pct_feriado:>13.1f}%                ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
##  Qualidade dos Dados

Integridade verificada automaticamente pelo pipeline antes de cada carga.
Registros com problemas são **isolados** e não afetam os KPIs abaixo.

In [0]:
df_silver = read_delta(SILVER_ITENS_VENDA_CAIXA_PATH, adls_options)
print(f"📋 Silver: {df_silver.count():,} linhas × {len(df_silver.columns)} colunas")
df_silver.printSchema()

In [0]:
df_dq = (
    read_delta(SILVER_DQ_METRICS_PATH, adls_options)
    .filter(col("tabela") == "physical_itens_venda_caixa")
    .groupBy("regra")
    .agg(
        spark_sum("qtd_afetados").alias("qtd_afetados"),
        spark_sum("qtd_total").alias("qtd_total"),
    )
    .orderBy("regra")
)
pdf_dq = df_dq.toPandas()

NOMES_REGRA = {
    "01_pk_id_item_venda_nula_ou_duplicada": "Integridade do ID do Item",
    "02_fk_id_transacao_invalida":           "Validade da Transação (FK)",
    "03_quantidade_invalida":                "Validade da Quantidade",
    "04_preco_invalido":                     "Validade do Preço",
    "05_valor_total_inconsistente":          "Consistência do Valor Total",
    "06_categoria_produto_nao_classificado": "Classificação Perecível/Seco",
}
pdf_dq["regra_negocio"] = pdf_dq["regra"].map(lambda x: NOMES_REGRA.get(x, x))
pdf_dq["pct_valido"]    = (
    100 - pdf_dq["qtd_afetados"] / pdf_dq["qtd_total"] * 100
).round(1)

# ── Score cards minimalistas ──────────────────────────────────────────────────
n   = len(pdf_dq)
fig = plt.figure(figsize=(4.0 * n, 3.2), facecolor="white")

for idx, (_, row) in enumerate(pdf_dq.iterrows()):
    pct   = row["pct_valido"]
    erros = int(row["qtd_afetados"])
    if pct >= 95:
        cor, status = COR_POSITIVO, "✓ Aprovado"
    elif pct >= 80:
        cor, status = COR_ALERTA,   "⚠ Atenção"
    else:
        cor, status = COR_NEGATIVO, "✗ Crítico"

    detalhe = f"{erros:,} erro(s)" if erros > 0 else "Sem erros"
    card_minimalista(fig, n, idx + 1, pct, f"{pct:.1f}%",
                     status, cor, row["regra_negocio"], detalhe)

fig.suptitle("Qualidade dos Dados — Itens de Venda (Silver)",
             fontsize=12, fontweight="bold", color="#212121", y=1.04)
plt.tight_layout(pad=1.2)
plt.show()

In [0]:
# ── Gráfico interativo — % válido por critério ────────────────────────────────
fig = go.Figure(go.Bar(
    x=pdf_dq["regra_negocio"],
    y=pdf_dq["pct_valido"],
    marker_color=[COR_POSITIVO if p >= 95 else (COR_ALERTA if p >= 80 else COR_NEGATIVO)
                  for p in pdf_dq["pct_valido"]],
    text=pdf_dq["pct_valido"].apply(lambda x: f"{x:.1f}%"),
    textposition="inside",
    textfont=dict(color="white", size=12, family="Arial Black"),
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Válidos: <b>%{y:.1f}%</b><br>"
        "Erros: <b>%{customdata:,}</b><extra></extra>"
    ),
    customdata=pdf_dq["qtd_afetados"],
    width=0.5,
))
fig.add_hline(y=95, line_dash="dot", line_color=COR_CINZA, line_width=1.5,
              annotation_text="Meta: 95%", annotation_font_color=COR_CINZA,
              annotation_position="top right")
fig.update_layout(
    **LAYOUT_BASE,
    title=dict(text="% de Registros Válidos por Critério de Qualidade",
               font=dict(size=14, color="#212121")),
    xaxis=dict(title="Critério", tickangle=-20,
               tickfont=dict(color="#424242", size=9)),
    yaxis=dict(title="% Válido", range=[0, 108], ticksuffix="%",
               gridcolor="#E0E0E0", tickfont=dict(color="#424242")),
    height=420,
    margin=dict(t=65, b=100, l=70, r=120),
)
fig.show()

---
##  KPI 6 — Receita por Produto por Loja por Mês
*Quais produtos geram mais receita e em quais lojas.*

In [0]:
df_kpi6 = (
    df_gold
    .filter(col("ano") == ano_ref)
    .groupBy("codigo_barras_produto", "ano", "mes")
    .agg(spark_round(spark_sum("receita_produto_loja_mes"), 2).alias("receita_total"))
    .orderBy(col("receita_total").desc())
)
pdf_kpi6 = df_kpi6.toPandas()

if not pdf_kpi6.empty:
    pdf_top10 = pdf_kpi6.head(10).copy()
    pdf_top10["produto_curto"] = pdf_top10["codigo_barras_produto"].str.split("-").str[0]

    top_prod = pdf_top10.iloc[0]
    print(f"🏆 Produto mais vendido ({ano_ref}): {top_prod['produto_curto']}")
    print(f"   Receita total: {formatar_reais(top_prod['receita_total'])}")

    # ── Plotly — Top 10 produtos por receita ─────────────────────────────────
    fig = go.Figure(go.Bar(
        y=pdf_top10["produto_curto"][::-1],
        x=pdf_top10["receita_total"][::-1],
        orientation="h",
        marker_color=COR_AZUL,
        marker_line=dict(color="white", width=0.5),
        text=pdf_top10["receita_total"][::-1].apply(formatar_reais),
        textposition="inside",
        textfont=dict(color="white", size=10),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Receita: <b>%{x:,.2f}</b><extra></extra>"
        ),
        width=0.65,
    ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text=f"Top 10 Categorias por Receita — {ano_ref}",
                   font=dict(size=14, color="#212121")),
        xaxis=dict(title="Receita (R$)", tickprefix="R$ ", tickformat=",.0f",
                   gridcolor="#F5F5F5", tickfont=dict(color="#424242")),
        yaxis=dict(title="", tickfont=dict(color="#212121", size=10)),
        height=420,
        margin=dict(t=65, b=60, l=180, r=40),
    )
    fig.show()

    # ── Matplotlib (estático) ─────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 5), facecolor="white")
    ax.barh(pdf_top10["produto_curto"][::-1], pdf_top10["receita_total"][::-1],
            color=COR_AZUL, edgecolor="white", height=0.6)
    ax.set_title(f"Top 10 Categorias por Receita — {ano_ref}",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Receita (R$)", color="#616161", fontsize=9)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}"))
    ax.tick_params(colors="#424242", labelsize=9)
    ax.grid(axis="x", color="#F5F5F5", linewidth=0.8)
    plt.tight_layout()
    plt.show()

---
##  KPI 7 — Top 10 Produtos por Loja por Trimestre
*Os produtos com maior participação de receita em cada loja.*

In [0]:
df_kpi7 = (
    df_gold
    .filter(col("rank_produto_trimestre").isNotNull()
            & (col("rank_produto_trimestre") <= 10)
            & (col("ano") == ano_ref))
    .select("codigo_barras_produto", "ano", "trimestre",
            "rank_produto_trimestre", "receita_trimestre")
    .distinct()
    .orderBy("trimestre", "rank_produto_trimestre")
)
pdf_kpi7 = df_kpi7.toPandas()

if not pdf_kpi7.empty:
    pdf_kpi7["produto_curto"] = pdf_kpi7["codigo_barras_produto"].str.split("-").str[0]
    pdf_kpi7["trimestre_fmt"] = pdf_kpi7["ano"].astype(str) + " T" + pdf_kpi7["trimestre"].astype(str)

    # ── Plotly — top 5 do trimestre mais recente ─────────────────────────────
    trim_ref = pdf_kpi7["trimestre"].max()
    pdf_trim = pdf_kpi7[pdf_kpi7["trimestre"] == trim_ref].head(10)

    fig = go.Figure(go.Bar(
        x=pdf_trim["produto_curto"],
        y=pdf_trim["receita_trimestre"],
        marker_color=[PALETA_DISCRETA[i % len(PALETA_DISCRETA)]
                      for i in range(len(pdf_trim))],
        marker_line=dict(color="white", width=0.5),
        text=pdf_trim["receita_trimestre"].apply(formatar_reais),
        textposition="inside",
        textfont=dict(color="white", size=9),
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Receita no trimestre: <b>%{y:,.2f}</b><br>"
            "Ranking: <b>#%{customdata}</b><extra></extra>"
        ),
        customdata=pdf_trim["rank_produto_trimestre"],
        width=0.65,
    ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text=f"Top 10 Produtos — {ano_ref} T{trim_ref}",
            font=dict(size=14, color="#212121"),
        ),
        xaxis=dict(title="Categoria do Produto", tickangle=-30,
                   tickfont=dict(color="#424242", size=9)),
        yaxis=dict(title="Receita no Trimestre (R$)", tickprefix="R$ ",
                   tickformat=",.0f", gridcolor="#F5F5F5",
                   tickfont=dict(color="#424242")),
        height=420,
        margin=dict(t=65, b=100, l=90, r=40),
    )
    fig.show()

    # ── Matplotlib (estático) ─────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4.5), facecolor="white")
    cores = [PALETA_DISCRETA[i % len(PALETA_DISCRETA)] for i in range(len(pdf_trim))]
    ax.bar(pdf_trim["produto_curto"], pdf_trim["receita_trimestre"],
           color=cores, edgecolor="white", width=0.6)
    ax.set_title(f"Top 10 Produtos — {ano_ref} T{trim_ref}",
                 fontsize=11, fontweight="bold")
    ax.set_ylabel("Receita no Trimestre (R$)", color="#616161", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right", fontsize=8)
    ax.tick_params(colors="#424242")
    ax.grid(axis="y", color="#F5F5F5", linewidth=0.8)
    plt.tight_layout()
    plt.show()

---
##  KPI 8 — Perecíveis vs Secos (MoM)
*Crescimento mensal da receita por categoria de produto.*

> **Nota:** classificação perecível/seco por inferência sobre o nome
> da categoria — ver limitação em `governanca/00_data_quality_rules`.

In [0]:
df_kpi8 = (
    df_gold
    .select("categoria_produto", "ano", "mes",
            "receita_categoria_mes", "crescimento_mom_categoria_pct")
    .distinct()
    .filter(col("categoria_produto").isNotNull()
            & (col("categoria_produto") != "NAO_CLASSIFICADO"))
    .orderBy("categoria_produto", "ano", "mes")
)
pdf_kpi8 = df_kpi8.toPandas()

CORES_CAT = {"perecivel": COR_POSITIVO, "seco": COR_AZUL}
NOMES_CAT = {"perecivel": "Perecíveis 🥩", "seco": "Secos 🥫"}
DESCR_CAT = {
    "perecivel": "Laticínios · Carnes · Hortifruti · Frios",
    "seco":      "Mercearia · Limpeza · Higiene · Bebidas",
}

pdf_kpi10 = None
pct_fer   = 0

if not pdf_kpi8.empty:
    pdf_kpi8["ano_mes"] = (
        pdf_kpi8["ano"].astype(str) + "-" +
        pdf_kpi8["mes"].astype(str).str.zfill(2)
    )

    # ── Receita acumulada por categoria no ano de referência ─────────────────
    pdf_cat_ano = (
        pdf_kpi8[pdf_kpi8["ano"] == ano_ref]
        .groupby("categoria_produto")["receita_categoria_mes"]
        .sum()
        .reset_index()
    )
    total_cat = pdf_cat_ano["receita_categoria_mes"].sum()
    for _, r in pdf_cat_ano.iterrows():
        pct = r["receita_categoria_mes"] / total_cat * 100 if total_cat else 0
        nome = NOMES_CAT.get(r["categoria_produto"], r["categoria_produto"])
        print(f"  {nome}: {formatar_reais(r['receita_categoria_mes'])} ({pct:.1f}% da receita {ano_ref})")

    # ── Score cards lado a lado ───────────────────────────────────────────────
    fig = plt.figure(figsize=(8.4, 3.2), facecolor="white")
    for i, cat in enumerate(["perecivel", "seco"]):
        row_cat = pdf_cat_ano[pdf_cat_ano["categoria_produto"] == cat]
        receita = float(row_cat["receita_categoria_mes"].values[0]) if not row_cat.empty else 0
        pct_cat = receita / total_cat * 100 if total_cat else 0
        cor     = CORES_CAT[cat]
        card_minimalista(
            fig, 2, i + 1,
            100,
            formatar_reais(receita),
            f"{pct_cat:.1f}% da receita {ano_ref}",
            cor,
            NOMES_CAT[cat],
            DESCR_CAT[cat],
            fontsize_valor=22,   # fonte menor para valores monetários longos
        )
    fig.suptitle(f"Receita Total por Categoria — {ano_ref}",
                 fontsize=12, fontweight="bold", color="#212121", y=1.04)
    plt.tight_layout(pad=1.5)
    plt.show()

    # ── Série temporal — Plotly interativo ───────────────────────────────────
    pdf_rede = (
        pdf_kpi8
        .groupby(["categoria_produto", "ano_mes"])["receita_categoria_mes"]
        .sum()
        .reset_index()
        .sort_values("ano_mes")
    )

    fig = go.Figure()
    for cat, grupo in pdf_rede.groupby("categoria_produto"):
        nome = NOMES_CAT.get(cat, cat)
        fig.add_trace(go.Scatter(
            x=grupo["ano_mes"],
            y=grupo["receita_categoria_mes"],
            name=nome,
            mode="lines+markers",
            line=dict(color=CORES_CAT.get(cat, COR_CINZA), width=2.5),
            marker=dict(size=6),
            hovertemplate=(
                f"<b>{nome}</b><br>"
                "Mês: <b>%{x}</b><br>"
                "Receita: <b>R$ %{y:,.2f}</b><extra></extra>"
            ),
        ))
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(
            text=f"Receita Mensal — Perecíveis vs Secos ({periodo_hist})",
            font=dict(size=14, color="#212121"),
        ),
        xaxis=dict(
            title="Mês/Ano",
            tickangle=-45,
            tickfont=dict(color="#424242", size=9),
            gridcolor="#F5F5F5",
        ),
        yaxis=dict(
            title="Receita (R$)",
            tickprefix="R$ ",
            tickformat=",.0f",
            gridcolor="#F5F5F5",
            tickfont=dict(color="#424242"),
        ),
        height=420,
        hovermode="x unified",
        legend=dict(
            orientation="h", yanchor="bottom", y=1.02,
            xanchor="left", x=0, font=dict(size=11),
        ),
        margin=dict(t=90, b=80, l=90, r=40),
    )
    fig.show()

    # ── Crescimento MoM — Plotly (último mês disponível) ─────────────────────
    pdf_mom = (
        pdf_kpi8[pdf_kpi8["crescimento_mom_categoria_pct"].notna()]
        .sort_values(["ano", "mes"])
    )
    if not pdf_mom.empty:
        ultimo_ano = pdf_mom["ano"].iloc[-1]
        ultimo_mes = pdf_mom["mes"].iloc[-1]
        pdf_mom_ref = pdf_mom[
            (pdf_mom["ano"] == ultimo_ano) &
            (pdf_mom["mes"] == ultimo_mes)
        ].copy()

        # Insight automático
        for _, r in pdf_mom_ref.iterrows():
            nome = NOMES_CAT.get(r["categoria_produto"], r["categoria_produto"])
            sinal = "▲" if r["crescimento_mom_categoria_pct"] >= 0 else "▼"
            print(f"  {nome}: {sinal} {r['crescimento_mom_categoria_pct']:+.1f}% vs mês anterior")

        v_max = pdf_mom_ref["crescimento_mom_categoria_pct"].abs().max()
        x_range = [-v_max * 1.4, v_max * 1.4]

        fig2 = go.Figure(go.Bar(
            y=[NOMES_CAT.get(c, c) for c in pdf_mom_ref["categoria_produto"]],
            x=pdf_mom_ref["crescimento_mom_categoria_pct"],
            orientation="h",
            marker_color=[COR_POSITIVO if v >= 0 else COR_NEGATIVO
                          for v in pdf_mom_ref["crescimento_mom_categoria_pct"]],
            marker_line=dict(color="white", width=0.5),
            text=pdf_mom_ref["crescimento_mom_categoria_pct"].apply(
                lambda x: f"{x:+.1f}%"
            ),
            textposition="outside",
            textfont=dict(
                color=[COR_POSITIVO if v >= 0 else COR_NEGATIVO
                       for v in pdf_mom_ref["crescimento_mom_categoria_pct"]],
                size=13, family="Arial Black",
            ),
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Crescimento MoM: <b>%{x:+.1f}%</b><extra></extra>"
            ),
            width=0.4,
        ))
        fig2.add_vline(x=0, line_color="#212121", line_width=1.5)
        fig2.update_layout(
            **LAYOUT_BASE,
            title=dict(
                text=f"Crescimento MoM — {ultimo_ano}/{str(ultimo_mes).zfill(2)} vs mês anterior",
                font=dict(size=14, color="#212121"),
            ),
            xaxis=dict(
                title="Variação (%)",
                ticksuffix="%",
                range=x_range,
                gridcolor="#F5F5F5",
                zeroline=True, zerolinecolor="#212121", zerolinewidth=1.5,
                tickfont=dict(color="#424242"),
            ),
            yaxis=dict(title="", tickfont=dict(color="#212121", size=11)),
            height=320,
            margin=dict(t=65, b=60, l=140, r=100),
        )
        fig2.show()

    # ── Matplotlib (estático para PR) ─────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4.5), facecolor="white")
    for cat, grupo in pdf_rede.groupby("categoria_produto"):
        g = grupo.sort_values("ano_mes")
        ax.plot(g["ano_mes"], g["receita_categoria_mes"],
                marker="o", markersize=4, linewidth=2.5,
                label=NOMES_CAT.get(cat, cat),
                color=CORES_CAT.get(cat, COR_CINZA))
    ax.set_title(
        f"Receita Mensal — Perecíveis vs Secos ({periodo_hist})",
        fontsize=11, fontweight="bold",
    )
    ax.set_xlabel("Mês/Ano", color="#616161", fontsize=9)
    ax.set_ylabel("Receita (R$)", color="#616161", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"R$ {x:,.0f}"))
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.legend(fontsize=10, framealpha=0.9, loc="upper left")
    ax.grid(axis="y", color="#F5F5F5", linewidth=0.8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

else:
    print("⚠️  Sem dados de classificação perecível/seco disponíveis.")

---
##  KPI 9 — Média de Itens por Transação por Loja
*"Tamanho do carrinho" — quantos itens diferentes o cliente compra por visita.*

In [0]:
df_kpi9 = (
    df_gold
    .select("ano", "mes", "media_itens_transacao")
    .distinct()
    .filter(col("media_itens_transacao").isNotNull())
    .orderBy("ano", "mes")
)
pdf_kpi9 = df_kpi9.toPandas()

if not pdf_kpi9.empty:
    pdf_kpi9["ano_mes"] = (
        pdf_kpi9["ano"].astype(str) + "-" +
        pdf_kpi9["mes"].astype(str).str.zfill(2)
    )
    media_geral = pdf_kpi9["media_itens_transacao"].mean()
    max_mes     = pdf_kpi9.loc[pdf_kpi9["media_itens_transacao"].idxmax(), "ano_mes"]
    min_mes     = pdf_kpi9.loc[pdf_kpi9["media_itens_transacao"].idxmin(), "ano_mes"]

    print(f"🛒 Média geral de itens/transação: {media_geral:.2f}")
    print(f"   📈 Mês com mais itens  : {max_mes} ({pdf_kpi9['media_itens_transacao'].max():.2f})")
    print(f"   📉 Mês com menos itens : {min_mes} ({pdf_kpi9['media_itens_transacao'].min():.2f})")

    # ── Score card ────────────────────────────────────────────────────────────
    tendencia = pdf_kpi9["media_itens_transacao"].iloc[-1] >= media_geral
    cor_card  = COR_POSITIVO if tendencia else COR_ALERTA
    status    = "▲ Acima da média" if tendencia else "▼ Abaixo da média"

    fig = plt.figure(figsize=(4.2, 3.2), facecolor="white")
    card_minimalista(
        fig, 1, 1,
        100, f"{media_geral:.2f}",
        "Itens por transação",
        COR_AZUL, "Média da Rede",
        f"Referência: {periodo_hist}",
    )
    fig.suptitle("KPI 9 — Carrinho Médio",
                 fontsize=12, fontweight="bold", color="#212121", y=1.04)
    plt.tight_layout(pad=1.2)
    plt.show()

    # ── Plotly — evolução temporal ────────────────────────────────────────────
    pdf_sorted = pdf_kpi9.sort_values("ano_mes")
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=pdf_sorted["ano_mes"],
        y=pdf_sorted["media_itens_transacao"],
        mode="lines+markers",
        line=dict(color=COR_AZUL, width=2.5),
        marker=dict(size=6, color=COR_AZUL),
        fill="tozeroy",
        fillcolor="rgba(13, 71, 161, 0.08)",
        hovertemplate="<b>%{x}</b><br>Média: <b>%{y:.2f} itens</b><extra></extra>",
        name="Média de Itens",
    ))
    fig.add_hline(
        y=media_geral,
        line_dash="dot", line_color=COR_CINZA, line_width=1.5,
        annotation_text=f"Média geral: {media_geral:.2f}",
        annotation_font_color=COR_CINZA,
        annotation_position="top right",
    )
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text=f"Média de Itens por Transação — {periodo_hist}",
                   font=dict(size=14, color="#212121")),
        xaxis=dict(title="Mês/Ano", tickangle=-45,
                   tickfont=dict(color="#424242", size=9),
                   gridcolor="#F5F5F5"),
        yaxis=dict(title="Itens por Transação", gridcolor="#F5F5F5",
                   tickfont=dict(color="#424242")),
        height=400,
        showlegend=False,
        margin=dict(t=65, b=80, l=80, r=120),
    )
    fig.show()

    # ── Matplotlib (estático) ─────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4), facecolor="white")
    ax.plot(pdf_sorted["ano_mes"], pdf_sorted["media_itens_transacao"],
            color=COR_AZUL, linewidth=2.5, marker="o", markersize=5)
    ax.fill_between(pdf_sorted["ano_mes"],
                    pdf_sorted["media_itens_transacao"],
                    alpha=0.08, color=COR_AZUL)
    ax.axhline(media_geral, color=COR_CINZA, linestyle="--",
               linewidth=1.2, label=f"Média geral: {media_geral:.2f}")
    ax.set_title(f"Média de Itens por Transação — {periodo_hist}",
                 fontsize=11, fontweight="bold")
    ax.set_ylabel("Itens por Transação", color="#616161", fontsize=9)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(axis="y", color="#F5F5F5", linewidth=0.8)
    plt.tight_layout()
    plt.show()

---
##  KPI 10 — Vendas em Feriados
*Impacto dos feriados nacionais na receita das lojas físicas.*

In [0]:
df_kpi10 = (
    df_gold
    .filter(col("ano") == ano_ref)
    .groupBy("venda_em_feriado")
    .agg(
        spark_round(spark_sum("receita_produto_loja_mes"), 2).alias("receita"),
        count("*").alias("qtd_itens"),
    )
)
pdf_kpi10 = df_kpi10.toPandas()

if not pdf_kpi10.empty:
    total_rec   = pdf_kpi10["receita"].sum()
    rec_feriado = pdf_kpi10[pdf_kpi10["venda_em_feriado"] == True]["receita"].sum()
    rec_normal  = pdf_kpi10[pdf_kpi10["venda_em_feriado"] == False]["receita"].sum()
    pct_fer     = round(rec_feriado / total_rec * 100, 1) if total_rec else 0
    print(f"🗓️  Receita em dias normais  : {formatar_reais(rec_normal)} ({100-pct_fer:.1f}%)")
    print(f"   Receita em feriados     : {formatar_reais(rec_feriado)} ({pct_fer:.1f}%)")

    # ── Score cards ───────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(8.4, 3.2), facecolor="white")
    card_minimalista(
        fig, 2, 1,
        100, f"{pct_fer:.1f}%",
        "da receita total",
        COR_ALERTA if pct_fer > 5 else COR_AZUL,
        "Receita em Feriados",
        formatar_reais(rec_feriado),
    )
    card_minimalista(
        fig, 2, 2,
        100, f"{100-pct_fer:.1f}%",
        "da receita total",
        COR_AZUL,
        "Receita em Dias Normais",
        formatar_reais(rec_normal),
    )
    fig.suptitle(f"Impacto de Feriados — {ano_ref}",
                 fontsize=12, fontweight="bold", color="#212121", y=1.04)
    plt.tight_layout(pad=1.2)
    plt.show()

    # ── Plotly — Donut chart ──────────────────────────────────────────────────
    fig = go.Figure(go.Pie(
        labels=["Dias Normais", "Feriados"],
        values=[rec_normal, rec_feriado],
        hole=0.55,
        marker=dict(
            colors=[COR_AZUL, COR_ALERTA],
            line=dict(color="white", width=2),
        ),
        textinfo="label+percent",
        textfont=dict(size=12, color="white"),
        hovertemplate=(
            "<b>%{label}</b><br>"
            "Receita: <b>R$ %{value:,.2f}</b><br>"
            "Participação: <b>%{percent}</b><extra></extra>"
        ),
    ))
    fig.add_annotation(
        text=f"<b>{pct_fer:.1f}%</b><br>feriados",
        x=0.5, y=0.5, showarrow=False,
        font=dict(size=14, color="#212121"),
    )
    fig.update_layout(
        **LAYOUT_BASE,
        title=dict(text=f"Participação de Feriados na Receita — {ano_ref}",
                   font=dict(size=14, color="#212121")),
        height=380,
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=-0.15,
                    xanchor="center", x=0.5, font=dict(size=10)),
        margin=dict(t=65, b=60, l=40, r=40),
    )
    fig.show()

    # ── Matplotlib (estático) ─────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(6, 4), facecolor="white")
    wedges, texts, autotexts = ax.pie(
        [rec_normal, rec_feriado],
        labels=["Dias Normais", "Feriados"],
        colors=[COR_AZUL, COR_ALERTA],
        autopct="%1.1f%%",
        startangle=90,
        wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2),
        pctdistance=0.75,
    )
    for t in autotexts:
        t.set_color("white")
        t.set_fontweight("bold")
        t.set_fontsize(11)
    ax.set_title(f"Receita: Dias Normais vs Feriados — {ano_ref}",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
##  Painel de Alertas
*Itens e categorias que requerem atenção da gestão.*

In [0]:
print("=" * 66)
print("   🚨  PAINEL DE ALERTAS — ITENS DE VENDA")
print("=" * 66)

alertas = []

# Alerta 1: Queda MoM em perecíveis
if not pdf_kpi8.empty:
    pdf_mom_alert = (
        pdf_kpi8[pdf_kpi8["crescimento_mom_categoria_pct"].notna()]
        .sort_values(["ano", "mes"])
    )
    if not pdf_mom_alert.empty:
        ultimo_a = pdf_mom_alert["ano"].iloc[-1]
        ultimo_m = pdf_mom_alert["mes"].iloc[-1]
        for _, row in pdf_mom_alert[
            (pdf_mom_alert["ano"] == ultimo_a) &
            (pdf_mom_alert["mes"] == ultimo_m) &
            (pdf_mom_alert["crescimento_mom_categoria_pct"] < -5)
        ].iterrows():
            cat = NOMES_CAT.get(row["categoria_produto"], row["categoria_produto"])
            alertas.append((
                "🔴 CRÍTICO", cat,
                "Crescimento MoM",
                f"Queda de {abs(row['crescimento_mom_categoria_pct']):.1f}% vs mês anterior",
            ))

# Alerta 2: Feriados com receita acima de 10%
if pdf_kpi10 is not None and not pdf_kpi10.empty and pct_fer > 10:
    alertas.append((
        "🟡 ATENÇÃO", "Toda a rede",
        "Concentração em feriados",
        f"{pct_fer:.1f}% da receita ocorre em feriados — avaliar impacto em dias normais",
    ))

# Alerta 3: Carrinho médio abaixo de 3 itens
if not pdf_kpi9.empty:
    ult_media = pdf_kpi9["media_itens_transacao"].iloc[-1]
    if ult_media < 3:
        alertas.append((
            "🟡 ATENÇÃO", "Toda a rede",
            "Carrinho médio baixo",
            f"Média de {ult_media:.2f} itens/transação no último mês — meta sugerida: ≥ 3",
        ))

if alertas:
    for nivel, entidade, indicador, detalhe in alertas:
        print(f"\n  {nivel}")
        print(f"  {'Entidade':<12}: {entidade}")
        print(f"  {'Indicador':<12}: {indicador}")
        print(f"  {'Detalhe':<12}: {detalhe}")
        print(f"  {'-'*60}")
else:
    print("\n  ✅ Nenhum alerta identificado.")
    print("     Todos os indicadores dentro do esperado.")

print(f"\n  Total de alertas: {len(alertas)}")
print("=" * 66)

---
## 📋 Próximos Passos

| # | Ação | Responsável | Prazo |
|---|------|-------------|-------|
| 1 | Validar alertas de queda MoM com gerentes comerciais | Gestão Comercial | Imediato |
| 2 | Avaliar mix de produtos perecíveis vs secos por loja | Operações | 15 dias |
| 3 | Analisar estratégia para dias de feriado | Marketing | 30 dias |
| 4 | Dashboard Looker conectado a `squad3.gold_physical_itens_venda_caixa` | Squad 3 | Próxima sprint |

---
> **Fonte dos dados:** Azure Data Lake (Raw → Bronze → Silver → Gold)
> **Tabela SQL Server:** `squad3.gold_physical_itens_venda_caixa`
> **Período:** {periodo_hist}